## Data Cleaning

*Cleaning process assisted by AI (Claude) for debugging and guidance.*

1. Fixed a duplicate header row that had been embedded as a data row
2. Standardized column names to lowercase, consistent formatting
3. Converted `year` to a numeric type
4. Distinguished two types of missing data: statistically suppressed (`<5`) values vs. genuinely blank entries
5. Preserved original values (`value` column) alongside a cleaned numeric version (`value_numeric`)
6. Identified and removed 269 duplicate rows

In [12]:
import pandas as pd

df = pd.read_excel('.../data/raw/data_glotip.xlsx')

df.columns = ['Iso3_code','country','region','subregion','indicator','dimension','category','sex','age','year','unit of measurement','value','source']

df.head()

,Iso3_code,country,region,subregion,indicator,dimension,category,sex,age,year,unit of measurement,value,source
0,28/10/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Iso3_code,Country,Region,Subregion,Indicator,Dimension,Category,Sex,Age,Year,Unit of measurement,txtVALUE,Source
2,ABW,Aruba,Americas,Latin America and the Caribbean,Detected trafficking victims,by country of repatriation,Ukraine,Total,Total,2010,Counts,<5,GLOTIP
3,AFG,Afghanistan,Asia,Southern Asia,Detected trafficking victims,by country of repatriation,Abroad,Total,Total,2003,Counts,<5,GLOTIP
4,AFG,Afghanistan,Asia,Southern Asia,Detected trafficking victims,by country of repatriation,Abroad,Total,Total,2008,Counts,<5,GLOTIP


In [13]:
print(df.info()) 

<class 'pandas.DataFrame'>
RangeIndex: 65400 entries, 0 to 65399
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Iso3_code            65400 non-null  str   
 1   country              65399 non-null  str   
 2   region               65399 non-null  str   
 3   subregion            65399 non-null  str   
 4   indicator            65399 non-null  str   
 5   dimension            65399 non-null  str   
 6   category             65399 non-null  str   
 7   sex                  65399 non-null  str   
 8   age                  65399 non-null  str   
 9   year                 65399 non-null  object
 10  unit of measurement  65399 non-null  str   
 11  value                65399 non-null  str   
 12  source               65399 non-null  str   
dtypes: object(1), str(12)
memory usage: 6.5+ MB
None


In [14]:
print(df.shape)
df['year'].apply(type).value_counts()

(65400, 13)


year
<class 'int'>      65398
<class 'float'>        1
<class 'str'>          1
Name: count, dtype: int64

In [15]:
df['year'].isna().sum()
df_clean = df.dropna(subset=['year'])

In [16]:
print(df.shape)
print(df_clean.shape)

(65400, 13)
(65399, 13)


In [17]:
df[df['year'].apply(lambda x: isinstance(x, str))]

,Iso3_code,country,region,subregion,indicator,dimension,category,sex,age,year,unit of measurement,value,source
1,Iso3_code,Country,Region,Subregion,Indicator,Dimension,Category,Sex,Age,Year,Unit of measurement,txtVALUE,Source


In [18]:
df_clean = df[df['year'].apply(lambda x: not isinstance(x, str))]
print(df_clean.shape)

(65399, 13)


In [19]:
df_clean['year'] = pd.to_numeric(df_clean['year'])
df_clean['year'].dtype

dtype('float64')

In [20]:
print(df_clean['year'].min(), '-', df_clean['year'].max())

2003.0 - 2023.0


In [21]:
df_clean['suppressed'] = df_clean['value'] == '<5'
df_clean['value_numeric'] = pd.to_numeric(df_clean['value'], errors='coerce')
print(f"Suppressed rows: {df_clean['suppressed'].sum()} ({df_clean['suppressed'].mean():.1%})")
df_clean[['value', 'suppressed', 'value_numeric']].head(10)

Suppressed rows: 16191 (24.8%)


,value,suppressed,value_numeric
0,NaN,False,NaN
2,<5,True,NaN
3,<5,True,NaN
4,<5,True,NaN
5,103,False,103.0
6,167,False,167.0
7,0,False,0.0
8,0,False,0.0
9,0,False,0.0
10,0,False,0.0


In [22]:
print(f"Suppressed (<5): {df_clean['suppressed'].sum()}")
print(f"Genuinely missing (blank in source): {df_clean['value'].isna().sum()}")

Suppressed (<5): 16191
Genuinely missing (blank in source): 1


In [25]:
dupes = df_clean[df_clean.duplicated(keep=False)]
print(dupes.groupby('country').size().sort_values(ascending=False).head(10))

country
Switzerland                                   32
Jordan                                        29
Mongolia                                      20
Thailand                                      17
Belize                                        17
Brunei Darussalam                             16
China, Macao Special Administrative Region    16
Ecuador                                       15
Solomon Islands                               14
Tonga                                         14
dtype: int64


In [27]:
df_clean = df_clean.drop_duplicates()
print(df_clean.shape)

(65130, 15)


In [29]:
# Rows where value_numeric is NaN but it's NOT suppressed and NOT genuinely blank
mystery_rows = df_clean[
    (df_clean['value_numeric'].isna()) & 
    (df_clean['suppressed'] == False) & 
    (df_clean['value'].notna())
]
print(mystery_rows.shape)
print(mystery_rows['value'].unique())

(814, 15)
<StringArray>
['6,112,019',     '1,757',     '2,004',     '2,386',     '2,948',     '1,981',
     '1,122',     '1,200',     '1,482',     '1,531',
 ...
     '2,744',     '9,645',     '6,442',     '7,491',     '8,067',     '1,025',
     '1,060',     '1,009',     '1,050',     '1,698']
Length: 651, dtype: str


In [30]:
df_clean['value_numeric'] = pd.to_numeric(
    df_clean['value'].str.replace(',', ''), 
    errors='coerce'
)

In [32]:
suppressed_count = df_clean['suppressed'].sum()
blank_count = df_clean['value'].isna().sum()
value_numeric_missing = df_clean['value_numeric'].isna().sum()

print(f"Suppressed: {suppressed_count}")
print(f"Genuinely blank: {blank_count}")
print(f"Sum: {suppressed_count + blank_count}")
print(f"value_numeric missing: {value_numeric_missing}")
print(f"Match?: {suppressed_count + blank_count == value_numeric_missing}")

Suppressed: 16097
Genuinely blank: 1
Sum: 16098
value_numeric missing: 16098
Match?: True


In [33]:
print("=== FINAL CLEANLINESS CHECK ===\n")

print(f"Shape: {df_clean.shape}")
print(f"Duplicate rows remaining: {df_clean.duplicated().sum()}")
print()

print("Column dtypes:")
print(df_clean.dtypes)
print()

print(f"Year range: {df_clean['year'].min()} - {df_clean['year'].max()}")
print(f"Any negative value_numeric?: {(df_clean['value_numeric'] < 0).any()}")
print()

print("Missing values per column:")
print(df_clean.isna().sum())
print()

print(f"Suppressed count matches flag?: {(df_clean['suppressed'].sum() == (df_clean['value'] == '<5').sum())}")

=== FINAL CLEANLINESS CHECK ===

Shape: (65130, 15)
Duplicate rows remaining: 0

Column dtypes:
Iso3_code                  str
country                    str
region                     str
subregion                  str
indicator                  str
dimension                  str
category                   str
sex                        str
age                        str
year                   float64
unit of measurement        str
value                      str
source                     str
suppressed                bool
value_numeric          float64
dtype: object

Year range: 2003.0 - 2023.0
Any negative value_numeric?: False

Missing values per column:
Iso3_code                  0
country                    1
region                     1
subregion                  1
indicator                  1
dimension                  1
category                   1
sex                        1
age                        1
year                       1
unit of measurement        1
value         

In [36]:
df_clean.to_csv('cleaned_human_trafficking_data.csv', index=False)
dir('data/processed')

['__add__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmod__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'capitalize',
 'casefold',
 'center',
 'count',
 'encode',
 'endswith',
 'expandtabs',
 'find',
 'format',
 'format_map',
 'index',
 'isalnum',
 'isalpha',
 'isascii',
 'isdecimal',
 'isdigit',
 'isidentifier',
 'islower',
 'isnumeric',
 'isprintable',
 'isspace',
 'istitle',
 'isupper',
 'join',
 'ljust',
 'lower',
 'lstrip',
 'maketrans',
 'partition',
 'removeprefix',
 'removesuffix',
 'replace',
 'rfind',
 'rindex',
 'rjust',
 'rpartition',
 'rsplit',
 'rstrip',
 'split',
 'splitlines',
 'startswith',
 'stri